In [1]:
!pip install -q transformers datasets torchaudio

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForCTC, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "facebook/mms-1b-all"

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.set_target_lang("ara")
model = AutoModelForCTC.from_pretrained(model_id)

model.load_adapter("ara")

model.to(device)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.ara.safetensors:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk
# Please make sure you have added the shared Google Drive folder to your 'My Drive'
# If the folder name is 'final' in your 'My Drive', the path should be:
DATASET_PATH  = "/content/drive/My Drive/final"
dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

print(f"\nTrain : {len(dataset['train']):,} segments")
print(f"Test  : {len(dataset['test']):,}  segments")
print(f"\nFeatures: {dataset['train'].features}")

Mounted at /content/drive
DatasetDict({
    train: Dataset({
        features: ['audio_id', 'audio', 'transcript', 'duration_s', 'seg_start', 'seg_end'],
        num_rows: 72314
    })
    test: Dataset({
        features: ['audio_id', 'audio', 'transcript', 'duration_s', 'seg_start', 'seg_end'],
        num_rows: 1005
    })
})

Train : 72,314 segments
Test  : 1,005  segments

Features: {'audio_id': Value('string'), 'audio': Audio(sampling_rate=16000, decode=False, stream_index=None), 'transcript': Value('string'), 'duration_s': Value('float32'), 'seg_start': Value('float32'), 'seg_end': Value('float32')}


In [12]:
import re
train_dataset = dataset['train']
test_dataset = dataset['test']
# Si vous voulez compter le nombre de phrases contenant des chiffres
count = 0

for i in test_dataset:
    # On cherche s'il y a au moins un chiffre dans la transcription
    if re.search(r'[0-9]', i['transcript']):
        print(i['transcript'])
        count += 1

print(f"\nTotal de phrases avec chiffres : {count}")


Total de phrases avec chiffres : 0


In [4]:
test_dataset = dataset["test"].remove_columns([
    "audio_id",
    "seg_start",
    "seg_end"
])

print(test_dataset)

Dataset({
    features: ['audio', 'transcript', 'duration_s'],
    num_rows: 1005
})


In [5]:
from datasets import Audio

test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [6]:
!pip install -q evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.1 MB/s eta 0:00:00


In [10]:
import evaluate
from tqdm.auto import tqdm
import numpy as np
import time
import re

# =========================
# Metrics
# =========================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# =========================
# Cleaning function

# Prepare references
# =========================
references = test_dataset["transcript"]

# =========================
# Inference + timing
# =========================
predictions = []
latencies = []

print(f"Running inference on {len(test_dataset)} samples...")

start_total = time.time()

for x in tqdm(test_dataset, total=len(test_dataset)):

    input_audio = {
        "array": np.array(x["audio"]["array"]),
        "sampling_rate": 16000
    }

    start = time.time()
    result = pipe(input_audio)
    end = time.time()

    predictions.append(result["text"])
    latencies.append(end - start)

end_total = time.time()

# =========================
# Latency metrics
# =========================
total_time = end_total - start_total
avg_latency = total_time / len(predictions)
mean_latency = np.mean(latencies)
median_latency = np.median(latencies)

# =========================
# Optimized Total audio duration 🔥
# =========================
total_audio_duration = sum(test_dataset["duration_s"])

# =========================
# Real-Time Factor (RTF)
# =========================
rtf = total_time / total_audio_duration

# =========================
# Clean predictions & references
# =========================
# =========================
# Compute WER / CER
# =========================
final_wer = wer_metric.compute(predictions=predictions, references=references)
final_cer = cer_metric.compute(predictions=predictions, references=references)

# =========================
# Print results
# =========================
print("\n===== BENCHMARK RESULTS =====")
print(f"WER: {final_wer:.4f}")
print(f"CER: {final_cer:.4f}")
print(f"Total inference time: {total_time:.2f} sec")
print(f"Average latency per sample: {avg_latency:.4f} sec")
print(f"Mean latency: {mean_latency:.4f} sec")
print(f"Median latency: {median_latency:.4f} sec")
print(f"Total audio duration: {total_audio_duration:.2f} sec")
print(f"Real-Time Factor (RTF): {rtf:.4f}")

Running inference on 1005 samples...


  0%|          | 0/1005 [00:00<?, ?it/s]


===== BENCHMARK RESULTS =====
WER: 0.8604
CER: 0.3707
Total inference time: 217.44 sec
Average latency per sample: 0.2164 sec
Mean latency: 0.2106 sec
Median latency: 0.1825 sec
Total audio duration: 3956.92 sec
Real-Time Factor (RTF): 0.0550


In [11]:
import pandas as pd
results_df = pd.DataFrame({"Reference": references, "Prediction": predictions})
display(results_df.head(50))

,Reference,Prediction
0,ويرحمني و اياكم في الدنيا و الآخرة,و يرحمني وإيكم في الدنيا والآخرة
1,اشك اناهي الحكومة الي باش تدخل لك في اصلاحات خ...,شك آناه الحكومة لل بش تتخلك في إصلاحات خطيرة
2,وهو قاعد ستة شهر,وقووقعد ستشرم
3,اذن نحن بحاجة الى مجلس تاسيسي ان ينجم ينظم كل ...,لإذن نحن بحاجة إلى مجلس تأسيسي نج من اظم كل هذا
4,الادارة العامة للميترو طالبة خدامة و الادارة ا...,دار العامة للميتروط أبخدامةللدار العاملولوزارب...
5,صندوق عجب صباح الخير بربي حبيت نتكلم بربي شوفو...,قع جبباخير برب حبية متكلم برب الشفون اللأثوام ...
6,كيفاش تعملت قوانينو آش بيها المنظومة هذيكة متا...,فيشة عملة قوانيناشبه المنظومذيكطعلكنا من الحدش...
7,اقعد معايا سليم خلي ناخذوا جمال معانا بالتليفو...,مع يسيم خلخد جميل معن بالتليفون جميل
8,معانا محرز بالتليفون مرحبا,عان محرس بالتليفون مرح ب
9,كيفاش ينجم يكون القرار هذايا ناجح ويوصل للاقلا...,فش نجميكون القرار هذية ناجح ووصل للإقلاع الاتا...


In [9]:
results_df = pd.DataFrame({
    "Reference": references,
    "Prediction": predictions
})

results_df.to_csv("whisper_large_v3_turbo_results.csv", index=False)